In [ ]:
#Importing all the necessary libraries
import os
import sys
import pickle
import time

import jax
import jax.numpy as jnp
import numpy as np
from scipy.io import loadmat

import flax
from flax import linen as nn

import optax
import matplotlib.pyplot as plt
import matplotlib
from typing import Callable, List
import scipy
from tqdm import tqdm

from models import FNO1d, FNO2d
from utils import save_model_params, load_model_params
from utils import dataloader

In [ ]:
burgers = loadmat("Burger.mat")
outputs = burgers['output']
Ns, Nt, Nx = outputs.shape
print(f"Ns: {Ns}, Nt: {Nt}, Nx: {Nx}")

In [ ]:
inputs = outputs[:,0,:]
inputs.shape, outputs.shape

In [ ]:
# Create coordinate grids
xspan = jnp.linspace(0, 1, Nx)  # spatial domain
tspan = jnp.linspace(0, 1, Nt)  # temporal domain

print(xspan.shape, tspan.shape)

# Meshgrid to create 2D coordinate arrays
[T, X] = jnp.meshgrid(tspan, xspan, indexing='ij')

#T and X have (Nt, Nx)
T_tiled = jnp.tile(T[None,:,:], (Ns,1,1))
X_tiled = jnp.tile(X[None,:,:], (Ns,1,1))

# tile inputs
inputs_tiled = jnp.tile(inputs[:,None,:], (1, Nt, 1))

#Stack all
inputs_to_FNO = jnp.stack([inputs_tiled, T_tiled, X_tiled], axis=-1)
output_FNO = outputs[:,:,:,None]
print(inputs_to_FNO.shape, output_FNO.shape)

In [ ]:
#Free up some memory
del inputs_tiled, T_tiled, X_tiled

In [ ]:
#Create the FNO-2D model object
fno = FNO2d(in_channels = inputs_to_FNO.shape[-1],
            out_channels = output_FNO.shape[-1],
            modes1 = 16,
            modes2 = 16,
            width = 32,
            n_blocks = 4,
            activation = nn.activation.gelu,  
)

In [ ]:
model_fn = jax.jit(fno.apply)

In [ ]:
#Instantiate the model params
params = fno.init(jax.random.PRNGKey(42), inputs_to_FNO[0:1])

In [ ]:
seed = 78633
result_dir = "./FNO_full_rollout"
filename = f"best_model_params_FNO_16modes_{seed}.pkl"

In [ ]:
best_params = load_model_params(result_dir, filename)

In [ ]:
start_time = time.time()
pred_FNO = model_fn(best_params, inputs_to_FNO)
end_time = time.time()

print(f"Total inference time for {pred_FNO.shape[0]} samples: {end_time-start_time} secs")

In [ ]:
pred_FNO.shape, output_FNO.shape

In [ ]:
overall_rel_L2_err = np.linalg.norm(pred_FNO - output_FNO)/np.linalg.norm(output_FNO)
overall_rel_L2_err

In [ ]:
pred_FNO_ = pred_FNO.squeeze(axis=-1)
output_FNO_ = output_FNO.squeeze(axis=-1)

pred_FNO_.shape, output_FNO_.shape

In [ ]:
auto_reg_error = []

for i in range(Nt):
    err = np.linalg.norm(pred_FNO_[:,i,:] - output_FNO_[:,i,:])/np.linalg.norm(output_FNO[:,i,:])
    auto_reg_error.append(err)

In [ ]:
auto_reg_error[60], auto_reg_error[70], auto_reg_error[90], auto_reg_error[100]

In [ ]:
save=True
if save:
    np.save("FNO_full_rollout/u_pred.npy", pred_FNO_)

In [ ]:
idx=70

In [ ]:
error = np.abs(pred_FNO_ - output_FNO_)

plt.figure(figsize = (12,3))
    
plt.subplot(1,3,1)
contour1 = plt.contourf(tspan, xspan, pred_FNO_[idx, :, :].T, levels = 20, cmap = 'jet')
cbar1 = plt.colorbar()
cbar1.ax.tick_params(labelsize = 12)
plt.xlabel("t", fontsize = 14)
plt.ylabel("x", fontsize = 14)
plt.xticks(fontsize = 12)
plt.yticks(fontsize = 12)
plt.title("Predicted", fontsize = 16)

plt.subplot(1,3,2)
contour2 = plt.contourf(tspan, xspan, output_FNO_[idx, :, :].T, levels = 20, cmap = 'jet')
cbar2 = plt.colorbar()
cbar2.ax.tick_params(labelsize = 12)
plt.xlabel("t", fontsize = 14)
plt.ylabel("x", fontsize = 14)
plt.xticks(fontsize = 12)
plt.yticks(fontsize = 12)
plt.title("Actual", fontsize = 16)


plt.subplot(1,3,3)
contour3 = plt.contourf(tspan, xspan, error[idx, :, :].T, levels = 20, cmap = 'Wistia')
cbar3 = plt.colorbar()
cbar3.ax.tick_params(labelsize = 12)
plt.xlabel("t", fontsize = 14)
plt.ylabel("x", fontsize = 14)
plt.xticks(fontsize = 12)
plt.yticks(fontsize = 12)
plt.title("Error", fontsize = 16)

plt.tight_layout()
# plt.savefig(filepath + f"/Contour_plots_sidx{i}.jpeg", dpi = 800)
plt.show()

In [ ]:
auto_reg_error = []

for i in range(Nt):
    err = np.linalg.norm(pred_FNO_[:,i,:] - output_FNO_[:,i,:])/np.linalg.norm(output_FNO[:,i,:])
    auto_reg_error.append(err)

In [ ]:
save=True
if save:
    np.save(result_dir + "/auto_reg_error_FNO2D_FR.npy", auto_reg_error)

In [ ]:
#Separate into train and test datasets
Ntrain = int(0.8*Ns)
perm = jax.random.permutation(jax.random.PRNGKey(0), Ns)

train_idx = perm[:Ntrain]
test_idx = perm[Ntrain:]

train_x = jnp.take(inputs_to_FNO, train_idx, axis=0)
test_x = jnp.take(inputs_to_FNO, test_idx, axis=0)

train_y = jnp.take(output_FNO, train_idx, axis=0)
test_y = jnp.take(output_FNO, test_idx, axis=0)

print(f"train_x shape: {train_x.shape}, train_y shape: {train_y.shape}")
print(f"test_x shape: {test_x.shape}, test_y shape: {test_y.shape}")

In [ ]:
#Stop gradients for test_x and test_y
test_x = jax.lax.stop_gradient(test_x)
test_y = jax.lax.stop_gradient(test_y)

In [ ]:
#Free up some memory
del inputs_to_FNO, output_FNO

In [ ]:
#Create the FNO-2D model object
fno = FNO2d(in_channels = train_x.shape[-1],
            out_channels = train_y.shape[-1],
            modes1 = 8,
            modes2 = 8,
            width = 32,
            n_blocks = 4,
            activation = nn.activation.gelu,  
)

In [ ]:
model_fn = jax.jit(fno.apply)

In [ ]:
#Instantiate the model params
params = fno.init(jax.random.PRNGKey(42), train_x[0:1])

In [ ]:
@jax.jit
def loss_fn(params, x, y):
    y_pred = model_fn(params, x)
    loss = jnp.mean((y_pred - y) ** 2)
    return loss

@jax.jit
def make_step(params, opt_state, x, y, test_x, test_y):
    loss, grads = jax.value_and_grad(loss_fn)(params, x, y)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    val_loss = loss_fn(params, test_x, test_y)
    return params, opt_state, loss, val_loss

In [ ]:
lr = 1e-3
lr_scheduler = optax.schedules.exponential_decay(init_value=lr, transition_steps=2000, decay_rate=0.96)
optimizer = optax.adam(lr_scheduler)
opt_state = optimizer.init(params)

In [ ]:
loss_history = []
val_loss_history = []
batch_size = 32
shuffle_key = jax.random.PRNGKey(80)
epochs = int(2e3)
min_val_loss = jnp.inf

result_dir = "./FNO_full_rollout"
filename = "best_model_params.pkl"

In [ ]:
for epoch in tqdm(range(epochs)):
    shuffle_key, subkey = jax.random.split(shuffle_key)
    total_loss = 0
    total_val_loss = 0
    nbatches = 0

    for batch_x, batch_y in dataloader(subkey, train_x, train_y, batch_size):
        batch_x = jax.lax.stop_gradient(batch_x)
        batch_y = jax.lax.stop_gradient(batch_y)

        params, opt_state, loss, val_loss = make_step(params, opt_state, batch_x, batch_y, test_x, test_y)

        total_loss += loss
        total_val_loss += val_loss
        nbatches += 1

    loss = total_loss / nbatches
    val_loss = total_val_loss / nbatches

    if val_loss < min_val_loss:
        best_params = params
        min_val_loss = val_loss
        save_model_params(best_params, result_dir, filename=filename)

    loss_history.append(loss)
    val_loss_history.append(val_loss)

    if epoch % 5 == 0:
        print(f"Epoch: {epoch}, Train loss: {loss:.6f}, Val loss: {val_loss:.6f}")

In [ ]:
plt.figure(dpi = 130)
plt.semilogy(np.arange(epoch+1), loss_history, label = "Train loss")
plt.semilogy(np.arange(epoch+1), val_loss_history, label = "Test loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")

plt.tick_params(which = 'major', axis = 'both', direction = 'in', length = 6)
plt.tick_params(which = 'minor', axis = 'both', direction = 'in', length = 3.5)
plt.minorticks_on()

plt.grid(alpha = 0.3)
plt.legend(loc = 'best')
plt.savefig(result_dir + "/loss_plot.jpeg", dpi = 800)
plt.show()

In [ ]:
#Save the loss arrays
if True:
    np.save(result_dir + "/Train_loss.npy",loss_history)
    np.save(result_dir + "/Test_loss.npy",val_loss_history)